# ノートブック③ 動画/画像 → 3DGS ＋ 物理パラメータ付き USD

実写の**動画または複数画像**から、
①**3D Gaussian Splatting**（.ply）と ②**物理パラメータ（衝突・質量・摩擦・重力）付きの仮想空間 USD**
（Omniverse / Isaac Sim 互換）を生成する Real2Sim パイプラインです。

## 関門（ゲート）方式
| 関門 | 内容 | OpenAI キー |
|---|---|---|
| **関門1**（セル11） | 動画 → COLMAP → 3DGS 学習 → .ply | 不要 |
| **関門2**（セル16） | メッシュ抽出 → スケール校正 → 静的 USD → Genesis 球落下検証 | 不要（gpt 校正を除く） |
| **関門3**（セル22） | 物体分離 → GPT-4o 物性推定 → 動的 USD → 落下静止検証 | **必要** |

## 前提
- **GPU ランタイム必須**（T4 で動作。所要: 関門1 まで ~1h、全体 ~2〜3h）
- 入力: 静止したシーンを**カメラを動かして**撮った動画（1〜3分、ゆっくり横移動）または画像 50〜300 枚
  - 撮影のコツは `docs/recon_pipeline.md` 参照
- フェーズ2（関門3）には OpenAI API キー（Colab Secrets の `OPENAI_API_KEY`）

上から順にセルを実行してください。

In [ ]:
# === セル2: GPU 確認 + VRAM 総量 ===
!nvidia-smi

import torch                       # Colab プリインストールの PyTorch
assert torch.cuda.is_available(), (
    "CUDA GPU が見つかりません。「ランタイム」→「ランタイムのタイプを変更」で GPU を選択してください。"
)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3   # VRAM 総量 [GB]
print(f"GPU: {torch.cuda.get_device_name(0)} / VRAM {vram_gb:.1f} GB")
if vram_gb < 14:
    print("注意: VRAM が少なめです。セル10 の GS_CAP_MAX を 500000 に下げてください。")

In [ ]:
# === セル3: 本リポジトリの clone ===
import os                          # パス・環境変数操作用

REPO_URL = "https://github.com/ringo7pie/RoboGen-GenesisWorld.git"   # 本リポジトリの公開 URL
REPO_DIR = "/content/RoboGen-GenesisWorld"     # clone 先
os.environ["GSPLAT_DIR"] = "/content/gsplat"             # gsplat リポジトリの配置先
os.environ["GSPLAT_PARSER_DIR"] = "/content/gsplat_parser"   # COLMAP パーサの隔離先

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull     # 既存 clone は最新化（修正 push の取り込み）
SCRIPTS = f"{REPO_DIR}/scripts"    # スクリプト置き場（以降のセルで使用）

In [ ]:
# === セル4: セットアップ一括実行（5〜10分） ===
# 依存インストール + gsplat clone (v1.5.3 固定) + COLMAP パーサの隔離インストール
!bash {REPO_DIR}/scripts/setup_recon.sh

In [ ]:
# === セル5: セルフチェック（初回は gsplat の JIT コンパイルで 1〜3 分かかります） ===
# ParticleField スキーマ（3DGS の USD 標準格納）の対応可否もここで表示されます。
# NG がある場合は、このセルの出力全体をエラー報告として共有してください。
!python {SCRIPTS}/check_env.py --stage recon

In [ ]:
# === セル6: Drive マウント + 作業ディレクトリ（RUN_ID）決定 ===
import datetime, shutil, glob     # RUN_ID 生成・バックアップ用
from google.colab import drive    # Drive マウント用
drive.mount("/content/drive")

DRIVE_OUT = "/content/drive/MyDrive/recon_outputs"   # 成果物の永続保存先
RESUME_RUN_ID = ""                 # 前回の続きから再開する場合は RUN_ID を入れる（例 "20260707-120000"）

RUN_ID = RESUME_RUN_ID or datetime.datetime.now().strftime("%Y%m%d-%H%M%S")   # 実行 ID
WS = f"/content/recon_work/{RUN_ID}"                 # 作業ディレクトリ
os.makedirs(WS, exist_ok=True)


# 指定サブパス群を Drive の backup へコピーする（関門ごとに呼ぶ）
def backup_to_drive(*subpaths):
    for sp in subpaths:
        src = os.path.join(WS, sp)                   # コピー元
        dst = os.path.join(DRIVE_OUT, RUN_ID, "backup", sp)   # コピー先
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif os.path.isfile(src):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
    print(f"Drive にバックアップ: {DRIVE_OUT}/{RUN_ID}/backup")


if RESUME_RUN_ID:                  # 再開時: Drive のバックアップを作業DIRへ復元
    bk = os.path.join(DRIVE_OUT, RESUME_RUN_ID, "backup")    # バックアップの場所
    assert os.path.isdir(bk), f"バックアップがありません: {bk}"
    shutil.copytree(bk, WS, dirs_exist_ok=True)
    print(f"復元しました: {bk} → {WS}")
print(f"RUN_ID = {RUN_ID}\nWS = {WS}")

In [ ]:
# === セル7: 入力の取り込み（動画 / 画像フォルダ / 画像zip） ===
INPUT_MODE = "drive"               # "drive" = Drive 上のパスを指定 / "upload" = ここでアップロード
INPUT_PATH = "/content/drive/MyDrive/recon_inputs/scene.mp4"   # drive モード時の入力パス（★書き換えてください）

input_dir = os.path.join(WS, "input")            # 取り込み先
os.makedirs(input_dir, exist_ok=True)
if INPUT_MODE == "upload":
    from google.colab import files               # ブラウザからのアップロード
    up = files.upload()                          # ファイル選択ダイアログ
    assert up, "ファイルがアップロードされていません"
    name = list(up.keys())[0]                    # アップロードされたファイル名
    INPUT_PATH = os.path.join(input_dir, name)
    os.replace(name, INPUT_PATH)
assert os.path.exists(INPUT_PATH), f"入力が見つかりません: {INPUT_PATH}"
print(f"入力: {INPUT_PATH}")

In [ ]:
# === セル8: フレーム抽出（fps 抽出 + 重複除去 + ブレ除去 + 縮小） ===
FPS = 2                            # 動画からの抽出レート（枚/秒）
LONG_EDGE = 1280                   # 長辺の縮小サイズ [px]（VRAM 不足なら 960 に）

!python {SCRIPTS}/extract_frames.py --input "{INPUT_PATH}" --workspace {WS} \
    --fps {FPS} --long-edge {LONG_EDGE}

# --- サムネイル格子で目視確認（ブレ・暗さ・被写体の写り） ---
import matplotlib.pyplot as plt    # 表示用
frames = sorted(glob.glob(f"{WS}/frames/*.jpg"))   # 抽出済みフレーム
print(f"フレーム数: {len(frames)}")
fig, axes = plt.subplots(2, 6, figsize=(18, 5))
for ax, p in zip(axes.flat, frames[:: max(1, len(frames) // 12)][:12]):
    ax.imshow(plt.imread(p)); ax.set_title(os.path.basename(p), fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# === セル9: COLMAP による SfM（カメラポーズ推定。5〜15分） ===
# 登録率 <60% で警告、<30% で中止（撮影のコツが表示されます）
!python {SCRIPTS}/run_colmap.py --workspace {WS}

In [ ]:
# === セル10: 3DGS の学習（gsplat。7k≈15分 / 15k≈35分 @T4） ===
GS_ITERS = 15000                   # 学習ステップ数（プレビューは 7000、高品質は 30000）
GS_CAP_MAX = 1000000               # ガウシアン数上限（VRAM 不足なら 500000）
SCENE_PRESET = "indoor"            # "indoor"（部屋）/ "outdoor"（屋外: 遠景対策強め）

!python {SCRIPTS}/train_gsplat.py --workspace {WS} \
    --iters {GS_ITERS} --cap-max {GS_CAP_MAX} --preset {SCENE_PRESET}

In [ ]:
# === セル11:【関門1】3DGS .ply の確認 + Drive 保存 ===
import json                        # stats 読み込み用
gstats = json.load(open(f"{WS}/gsplat_stats.json"))   # 学習統計
print(f"ply: {gstats['ply']}\nガウシアン数: {gstats['num_gaussians']} / PSNR: {gstats['psnr']}")

# 評価用レンダ画像があれば表示（学習品質の目視確認）
vals = sorted(glob.glob(f"{WS}/gsplat/renders/*.png"))[:4]   # 評価レンダ
if vals:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(vals), figsize=(5 * len(vals), 4))
    for ax, p in zip((axes if len(vals) > 1 else [axes]), vals):
        ax.imshow(plt.imread(p)); ax.axis("off")
    plt.tight_layout(); plt.show()

backup_to_drive("gsplat/ply", "colmap_ws/sparse", "gsplat_stats.json", "colmap_stats.json", "frames_meta.json")
print("\n関門1 突破。ply は https://superspl.at/editor にドラッグ&ドロップすると3D表示できます")
print(f"（Drive の {DRIVE_OUT}/{RUN_ID}/backup/gsplat/ply/ からダウンロード）")

## フェーズ1: 静的な物理シーン（衝突メッシュ + 床 + 重力）

3DGS からシーンの衝突メッシュを抽出し、床平面を分離して、実寸スケール校正のうえ
**UsdPhysics 準拠の静的 USD**（+ 3DGS 入り）を生成 → Genesis で球を落として貫通なしを検証します。
ここまで OpenAI キーは不要です（スケール校正の gpt 方式を除く）。

In [ ]:
# === セル13: 衝突メッシュ抽出 + 床分離（深度レンダ → TSDF 融合。5〜10分） ===
!python {SCRIPTS}/extract_mesh.py --workspace {WS} --preset {SCENE_PRESET}

# --- 目盛り付き3面図の表示（次セルの手入力スケール校正の読み取り資料） ---
import matplotlib.pyplot as plt
pv = [f"{WS}/mesh/preview_{n}.png" for n in ("top", "front", "side")]   # 3面図
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, p in zip(axes, pv):
    ax.imshow(plt.imread(p)); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# === セル14: 実寸スケール校正（3方式切替） ===
SCALE_MODE = "manual"              # 校正方式: "manual" / "aruco" / "gpt"

# --- manual 用: 上の3面図の目盛りで既知の物の長さを読み取り、実寸と合わせる ---
MANUAL_REAL_DIST = 0.72            # その物の実測寸法 [m]（例: 机の高さ 0.72m）
MANUAL_COLMAP_DIST = 1.85          # 3面図から読み取った同じ寸法（COLMAP 単位）
# （2点指定方式を使う場合は下を設定して MANUAL_COLMAP_DIST = None にする）
MANUAL_POINTS = None               # 例 "frame=frame_00050.jpg;a=512,300;b=512,880"

# --- aruco 用: 撮影時に置いた ArUco マーカーの一辺実寸 ---
ARUCO_MARKER_SIZE = 0.15           # マーカー一辺 [m]
ARUCO_DICT = "DICT_4X4_50"         # マーカー辞書

if SCALE_MODE == "manual":
    if MANUAL_COLMAP_DIST:
        !python {SCRIPTS}/calibrate_scale.py --workspace {WS} --mode manual \
            --real-dist {MANUAL_REAL_DIST} --colmap-dist {MANUAL_COLMAP_DIST}
    else:
        !python {SCRIPTS}/calibrate_scale.py --workspace {WS} --mode manual \
            --real-dist {MANUAL_REAL_DIST} --points "{MANUAL_POINTS}"
elif SCALE_MODE == "aruco":
    !python {SCRIPTS}/calibrate_scale.py --workspace {WS} --mode aruco \
        --marker-size {ARUCO_MARKER_SIZE} --aruco-dict {ARUCO_DICT}
else:                              # gpt 方式（OpenAI キーが必要。誤差 ±20〜30%）
    from google.colab import userdata            # Colab Secrets の読み取り
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    !python {SCRIPTS}/calibrate_scale.py --workspace {WS} --mode gpt

In [ ]:
# === セル15: 静的 USD の生成（3DGS 入り・UsdPhysics 準拠） ===
!python {SCRIPTS}/build_usd.py --workspace {WS} --phase static --validate --usda

# --- 生成された USD のプリム階層を表示 ---
from pxr import Usd                # USD の読み込み確認
stage = Usd.Stage.Open(f"{WS}/usd/scene_static.usdc")
for prim in stage.Traverse():
    depth = str(prim.GetPath()).count("/") - 1   # 階層の深さ
    print("  " * depth + f"{prim.GetName()}  ({prim.GetTypeName()})")

In [ ]:
# === セル16:【関門2】Genesis 物理検証（球落下・貫通チェック）+ Drive 保存 ===
!python {SCRIPTS}/verify_genesis_usd.py --workspace {WS} --mode static

from IPython.display import Video  # 検証動画の表示
backup_to_drive("mesh", "scale", "usd", "verify")
Video(f"{WS}/verify/static_check.mp4", embed=True, width=640)

## フェーズ2: 物体の分離と物性付与（動的剛体）

テキストで指定した物体（椅子・箱など）を SAM2 で分離し、**GPT-4o が材質・密度・摩擦を推定**、
質量（密度×実寸体積）付きの**動的剛体**として USD に組み込みます。

事前準備: Colab 左側のカギアイコン（Secrets）で `OPENAI_API_KEY` を登録し、
「ノートブックからのアクセス」を ON にしてください（ノートブック②と共通）。

In [ ]:
# === セル18: OpenAI キー設定 + 疎通確認 ===
from google.colab import userdata  # Colab Secrets の読み取り用
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")   # 環境変数経由で各スクリプトへ
os.environ["RECON_LLM_MODEL"] = "gpt-4o"     # 物性推定・スケール推定に使うモデル

from openai import OpenAI          # 疎通確認（軽い呼び出し1回）
res = OpenAI().chat.completions.create(model=os.environ["RECON_LLM_MODEL"],
                                       messages=[{"role": "user", "content": "ping"}], max_tokens=5)
print(f"疎通 OK: model={os.environ['RECON_LLM_MODEL']}")

In [ ]:
# === セル19: 物体の分離（GroundingDINO + SAM2 + 視点投票。初回はモデル DL で+数分） ===
OBJECT_PROMPTS = "chair. box. bottle."   # 分離したい物体（英語・ピリオド区切り。写っている物に合わせて変更）

!python {SCRIPTS}/segment_objects.py --workspace {WS} --prompts "{OBJECT_PROMPTS}"

# --- 検出物体のクロップ画像を目視確認（誤検出があればプロンプトを変えて再実行） ---
import json, matplotlib.pyplot as plt
objs = json.load(open(f"{WS}/objects/objects.json"))["objects"]   # 物体一覧
print(f"検出物体: {len(objs)} 件")
fig, axes = plt.subplots(1, max(len(objs), 1), figsize=(4 * max(len(objs), 1), 4))
for ax, ob in zip((axes if len(objs) > 1 else [axes]), objs):
    ax.imshow(plt.imread(f"{WS}/objects/obj_{ob['id']}/crop.jpg"))
    ax.set_title(f"#{ob['id']} {ob['label']}\n(視点{ob['n_obs']}/GS{ob['n_gaussians']})", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# === セル20: GPT-4o による物性推定（材質・密度・摩擦・反発 → 質量計算） ===
# 出力の表を確認し、直したい値があれば {WS}/physics/physics.json を編集して
# セル21 以降だけ再実行すれば反映されます。
!python {SCRIPTS}/estimate_physics.py --workspace {WS}

In [ ]:
# === セル21: 動的剛体入り USD の生成 ===
!python {SCRIPTS}/build_usd.py --workspace {WS} --phase dynamic --validate --usda

In [ ]:
# === セル22:【関門3】Genesis 落下検証（10cm 落下 → 静止判定） ===
!python {SCRIPTS}/verify_genesis_usd.py --workspace {WS} --mode dynamic

from IPython.display import Video  # 検証動画の表示
backup_to_drive("objects", "physics", "usd", "verify")
Video(f"{WS}/verify/dynamic_check.mp4", embed=True, width=640)

In [ ]:
# === セル23: usdz パッケージ化 + 全成果物の Drive 保存 ===
phase = "dynamic" if os.path.isfile(f"{WS}/physics/physics.json") else "static"   # フェーズ2 実施有無で切替
!python {SCRIPTS}/build_usd.py --workspace {WS} --phase {phase} --usdz --validate

import shutil                      # 最終成果物のコピー用
final_dir = f"{DRIVE_OUT}/{RUN_ID}/final"        # 完成品の置き場
os.makedirs(final_dir, exist_ok=True)
for pat in ("usd/*.usdc", "usd/*.usda", "usd/*.usdz", "usd/gaussians.ply", "usd/report.json",
            "gsplat/ply/*.ply", "verify/*.mp4"):
    for p in glob.glob(f"{WS}/{pat}"):
        shutil.copy(p, final_dir)
print(f"最終成果物を保存しました: {final_dir}")
!ls -lh {final_dir}

## 完了 — 成果物と次の一歩

Drive の `recon_outputs/<RUN_ID>/final/` に以下が保存されています:

| ファイル | 内容 |
|---|---|
| `scene_static.usdc` / `scene_dynamic.usdc` | 物理パラメータ付き USD（3DGS 入り） |
| `scene.usdz` | 単一ファイルのパッケージ版 |
| `point_cloud_*.ply` | 3D Gaussian Splatting（[SuperSplat](https://superspl.at/editor) で閲覧可） |
| `*_check.mp4` | Genesis 物理検証の動画 |
| `report.json` | 格納方式（ParticleField/sidecar）・検証結果・注意事項 |

### Omniverse / Isaac Sim で開く
1. `scene_*.usdc`（または `scene.usdz`）をローカルへダウンロード
2. **Isaac Sim 6.0 以降**で開くと、3DGS（ParticleField）もネイティブ描画されます
3. Play ボタンで物理シミュレーション開始（重力・衝突・動的剛体が機能します）

詳細な手順・調整方法: `docs/recon_pipeline.md` / うまくいかない場合: `docs/troubleshooting.md`